In [ ]:
# install libraries
!pip install -q transformers datasets accelerate pandas scikit-learn kaggle

In [ ]:
!pip install -U "accelerate>=1.10.1"

In [ ]:
# Import Libraries
import os
import transformers
import accelerate
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

In [ ]:
# Download Dataset from Kaggle
df_raw = pd.read_csv("RickAndMortyScripts.csv")
df_raw.head()

In [ ]:
# Preparation of Conversation data
lines = df_raw["line"].dropna().astype(str).tolist()
contexted=[]
n_context=7

for i in range(n_context, len(lines)):
    context=lines[i-n_context:i]
    response=lines[i]
    conversation=""
    for line in context:
        conversation+=line+" "
    conversation+=response
    contexted.append(conversation)  

df=pd.DataFrame({"text": contexted})    
train_df, val_df=train_test_split(df, test_size=0.1,random_state=42)

In [ ]:
print(train_df.shape, val_df.shape)
train_df.head()

In [ ]:
# Loading Tokenizer and model
model_name = "microsoft/DialoGPT-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# DialoGPT/GPT2 has no pad token by default, so we use EOS token as padding
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))

In [ ]:
# Creating Pytorch Dataset
class RickDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=512):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": encoding["input_ids"].squeeze()
        }
train_dataset = RickDataset(train_df["text"].tolist(), tokenizer)
val_dataset = RickDataset(val_df["text"].tolist(), tokenizer)

In [ ]:
# Data Collextor
# mlm=False because DialoGPT is a causal language model, not masked language model
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
# Training Settings

training_args=TrainingArguments(
    output_dir="./rickbot-output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    learning_rate=5e-5,
    weight_decay=0.01,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available()
    
)

In [ ]:
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)

In [ ]:
# Training the model
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer.train()

In [ ]:
trainer.save_model("./rickbot-final")
tokenizer.save_pretrained("./rickbot-final")

In [ ]:
# Chat eith Rickbot

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("./rickbot-final")
model = AutoModelForCausalLM.from_pretrained("./rickbot-final").to(device)

chat_history_ids = None

for step in range(5):
    user_input = input(">> User: ")

    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"
    ).to(device)

    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=200,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
        do_sample=True,
        top_k=100,
        top_p=0.7,
        temperature=0.8
    )

    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print("RickBot:", response)